# W34 · 大规模并行训练与评估

> 阶段四第 4 周。本周回答两个问题：4096 个环境并行到底快到什么程度、为什么？
> 以及拿到 Isaac Lab 的默认 PPO 配置后，rsl_rl 的关键超参怎么调。

## 学习目标

1. 用 **Amdahl 定律**定量分析并行仿真加速比，找出吞吐瓶颈；
2. 设计并执行 16 vs 1024 vs 4096 环境的吞吐对比实验，正确读取 FPS/SPS 指标；
3. 理解 rsl_rl PPO 的 batch 结构（`num_envs × num_steps_per_env`）与关键超参；
4. 掌握评估与 checkpoint 导出（play.py、TorchScript/ONNX）的标准流程。

## 运行前提

- 「并行度示意分析」cell 只需项目环境（numpy + matplotlib），可本地运行 ✅
- 所有 Isaac 命令与 rsl_rl 配置 cell 需 **NVIDIA GPU + Isaac Lab**，保持未执行 ⚠️

## 1. 为什么并行是数量级的提升

Isaac Gym 论文（[Makoviychuk et al., 2021](https://arxiv.org/abs/2108.10470)）的核心实验：
同一块 GPU 同时跑物理仿真和神经网络，观测/动作零拷贝，相比「CPU 物理 + GPU 网络」
的流水线，PPO 训练提速 **1~2 个数量级**。Rudin et al.（2022）沿此路线让四足
「几分钟学会走路」（论文标题就叫 *Learning to Walk in Minutes*）。

但并行不是免费的，**Amdahl 定律**规定了加速比上限。设总工作量中可并行部分占 $p$，
用 $N$ 个并行单元，则

$$S(N) = \frac{1}{(1-p) + \dfrac{p}{N}}$$

当 $N \to \infty$ 时 $S \to \dfrac{1}{1-p}$：串行占比 $1-p$ 决定天花板。

对应到 Isaac Lab 训练循环：

- **可并行 $p$**：物理 step、观测组装、奖励/终止计算、网络前向反向（batch 维度随 env 数扩大）
- **难并行 $1-p$**：PPO 的 mini-batch 循环内 kernel launch 开销、optimizer.step、
  日志记录、checkpoint 保存、以及 Python 端的逐 iteration 串行调度

所以实际曲线是：env 数翻倍 → FPS 先近似线性增长 → 在显存或串行开销处饱和。
下面的 cell 画出示意曲线（**示意数据，非实测**，实测定量见练习 1）：

In [ ]:
# 运行前提：项目 .venv（numpy + matplotlib）。本 cell 可在本机运行。
import numpy as np
import matplotlib.pyplot as plt

fig, axes = plt.subplots(1, 2, figsize=(12, 4.5))

# --- 左图：Amdahl 定律：不同可并行占比 p 下的加速比曲线 ---
N = np.logspace(0, 4.5, 200)  # 并行单元数 1 ~ 30000
for p in (0.90, 0.99, 0.999):
    S = 1.0 / ((1 - p) + p / N)
    axes[0].loglog(N, S, label=f"p = {p}")
axes[0].axhline(1 / (1 - 0.99), color="gray", ls="--", lw=1)
axes[0].annotate("p=0.99 的天花板 = 100×", xy=(3, 95), fontsize=9, color="gray")
axes[0].set_xlabel("并行单元数 N（如并行环境数）")
axes[0].set_ylabel("加速比 S(N)")
axes[0].set_title("Amdahl 定律：串行部分决定天花板")
axes[0].legend()
axes[0].grid(True, which="both", alpha=0.3)

# --- 右图：示意：FPS 随并行环境数增长并饱和（示意数据，非实测！）---
num_envs = np.array([16, 64, 256, 1024, 2048, 4096, 8192])
peak_fps = 120_000          # 假想的 GPU 饱和吞吐（示意）
per_env_fps = 150           # 单环境串行基准（示意）
fps = peak_fps * (1 - np.exp(-num_envs / 1500)) + per_env_fps * np.exp(-num_envs / 1500) * num_envs / 16
axes[1].loglog(num_envs, fps, "o-", color="tab:orange")
axes[1].axhline(peak_fps, color="gray", ls="--", lw=1)
axes[1].annotate(f"饱和 ≈ {peak_fps:,} FPS（示意）", xy=(30, peak_fps * 1.1),
                 fontsize=9, color="gray")
for x, y in zip(num_envs, fps):
    axes[1].annotate(f"{y:,.0f}", xy=(x, y), textcoords="offset points",
                     xytext=(0, 6), fontsize=8, ha="center")
axes[1].set_xlabel("并行环境数 --num_envs")
axes[1].set_ylabel("训练 FPS（仿真步/秒，示意）")
axes[1].set_title("示意：吞吐随 env 数增长并饱和")
axes[1].grid(True, which="both", alpha=0.3)

plt.tight_layout()
plt.show()

print("读图要点：")
print("1) p=0.99 时加速比天花板只有 100×——串行 1% 决定了上限；")
print("2) 右图为示意：实测请跑练习 1 的对比实验，曲线形状取决于 GPU 型号与任务。")

## 2. 吞吐对比实验：怎么做才对

在有 GPU 的机器上，**控制变量**只改 `--num_envs`：

```bash
for N in 16 256 1024 4096; do
  ./isaaclab.sh -p scripts/reinforcement_learning/rsl_rl/train.py \
      --task=Isaac-Ant-v0 --num_envs=$N --headless --max_iterations 50
done
```

**记录指标**（终端与 TensorBoard 都有）：

| 指标 | 含义 | 怎么看 |
|------|------|--------|
| `fps` | 每秒仿真步数（≈ 吞吐 SPS） | 取 iteration 10 之后的稳定均值，前几个 iteration 有初始化开销 |
| `mean_reward` | 学习进度 | 吞吐实验里不必等收敛，50 iteration 够看曲线斜率 |
| `step time` / 各 phase 耗时 | rsl_rl 日志的分段计时 | 区分「物理慢」还是「学习慢」 |

**预期结果（定性）**：16→1024 时 fps 近似线性爬升；2048→4096 后趋缓；
在高端 GPU 上 Ant 级任务稳定 fps 可达 $10^4$–$10^5$ 量级。
**注意**：fps 高 ≠ 学得快——4096 env 的 PPO 每次更新吃 $4096 \times 24 \approx 10^5$
样本，batch 太大可能降低样本效率，这正是下一节超参要处理的事。

## 3. rsl_rl 的 PPO：batch 结构与关键超参

rsl_rl 是 on-policy runner，每个 iteration 的数据流：

$$\underbrace{N_{\text{env}} \times T_{\text{steps}}}_{\text{一次 rollout 的样本数}} \;\xrightarrow{\;\text{分成 } B_{\text{mini}}\text{ 份}\;}\; \underbrace{E_{\text{epoch}} \times B_{\text{mini}}}_{\text{梯度更新次数/iteration}}$$

- $N_{\text{env}}$ = `--num_envs`（环境侧）
- $T_{\text{steps}}$ = `num_steps_per_env`（默认 24）
- $B_{\text{mini}}$ = `num_mini_batches`（默认 4），$E_{\text{epoch}}$ = `num_learning_epochs`（默认 5）

例：4096 env × 24 steps = 98,304 样本/iteration；分成 4 个 mini-batch（各 24,576），
重复 5 个 epoch → 每 iteration 20 次梯度更新。

| 超参 | 作用 | 调整经验 |
|------|------|----------|
| `num_steps_per_env` | rollout 长度 | 任务时序长（导航）调大；太短学不到长期回报 |
| `num_mini_batches` | 显存与噪声权衡 | 显存爆了先加它；太大则梯度噪声小但更新慢 |
| `num_learning_epochs` | 数据复用 | PPO clip 保护下 5 是安全值；过大会 off-policy 漂移 |
| `learning_rate` / `schedule="adaptive"` | 步长 | adaptive 按 `desired_kl` 自动调节，通常不用动 |
| `gamma` / `lam` | 回报折扣 / GAE |  locomotion 常用 0.99 / 0.95；长 horizon 任务 gamma 可到 0.995+ |
| `entropy_coef` | 探索 | 收敛后仍有抖动可调低；过早收敛可调高 |
| `clip_param` | PPO 裁剪 | 0.2 是标准；策略震荡可降到 0.1 |
| `empirical_normalization` | 观测归一化 | 几乎总是开 |

下面的配置模板对照的是官方 Anymal 四足任务的典型取值：

In [ ]:
# 前提：Isaac Lab 环境（isaaclab_rl 包）；未执行。
# 文件：tasks/<your_task>/agents/rsl_rl_ppo_cfg.py —— 对照官方 Anymal 配置的典型取值
from isaaclab_rl.rsl_rl import (
    RslRlOnPolicyRunnerCfg,
    RslRlPpoActorCriticCfg,
    RslRlPpoAlgorithmCfg,
)

agent_cfg = RslRlOnPolicyRunnerCfg(
    seed=42,
    device="cuda:0",
    num_steps_per_env=24,          # T：每 env 每次 rollout 的步数
    max_iterations=1500,
    save_interval=50,              # 每 50 iteration 存一次 checkpoint
    experiment_name="anymal_c_rough",
    empirical_normalization=True,  # 观测在线归一化
    policy=RslRlPpoActorCriticCfg(
        init_noise_std=1.0,        # 初始探索噪声（动作 std）
        actor_hidden_dims=[512, 256, 128],
        critic_hidden_dims=[512, 256, 128],
        activation="elu",
    ),
    algorithm=RslRlPpoAlgorithmCfg(
        value_loss_coef=1.0,
        use_clipped_value_loss=True,
        clip_param=0.2,
        entropy_coef=0.005,
        num_learning_epochs=5,
        num_mini_batches=4,
        learning_rate=1.0e-3,
        schedule="adaptive",       # 按 desired_kl 自动调 lr
        gamma=0.99,
        lam=0.95,
        desired_kl=0.01,
        max_grad_norm=1.0,
    ),
)

# 心算检查（见正文公式）：
# 4096 env × 24 steps = 98,304 样本/iteration
# 98,304 / 4 mini_batch = 24,576 样本/次前向反向
# 5 epochs × 4 mini_batches = 20 次梯度更新/iteration

## 4. 评估与导出：训练完之后的标准动作

```bash
# ① 少量环境回放 + 渲染，肉眼评估行为质量
./isaaclab.sh -p scripts/reinforcement_learning/rsl_rl/play.py \
    --task=Isaac-Velocity-Rough-Anymal-C-v0 --num_envs=32 \
    --checkpoint logs/rsl_rl/<run_dir>/model.pt

# ② play 脚本同时导出部署用模型（TorchScript + ONNX，位于 run 目录的 exported/ 下）
#    部署侧不再需要 rsl_rl/Isaac Lab，普通 torch.onnx 运行时即可推理
```

评估的**统计学要点**（Capstone 会强制要求）：

- 成功率类指标要报**试验次数 n**（下周 notebook 会算 Wilson 置信区间）；
- 连续指标（速度跟踪误差、能耗）报**均值 ± 标准差**，且注明 episode 数；
- 评估环境与训练环境要**同随机种子范围但不同种子**——防止过拟合特定初始状态。

## 5. 小结

- 并行收益由 Amdahl 定律封顶；先找到串行瓶颈（日志分段计时）再谈优化；
- fps 是**吞吐**指标，不是**学习速度**指标；大 batch 时代样本效率要靠超参找回来；
- rsl_rl 的 batch 结构 = `num_envs × num_steps_per_env / num_mini_batches`；
- 训练 → play.py 回放 → TorchScript/ONNX 导出 → 统计化评估，是 Isaac Lab 的完整闭环。

---

## ✏️ 练习

1. **吞吐实测**（★★，约 1.5 小时，需 GPU）
   按第 2 节命令在 Ant 任务上测 `num_envs ∈ {16, 256, 1024, 4096}` 的稳定 fps，
   画 log-log 曲线并标注饱和点；把实测曲线与本 notebook 的示意曲线叠在一起对比。
   **交付物**：`journal/throughput_curve.png` + 100 字解读（饱和发生在哪、为什么）。
2. **Amdahl 反推**（★★，约 30 分钟）
   用练习 1 中 `N=16` 与 `N=4096` 的 fps，代入 $S(N)=1/((1-p)+p/N)$ 反推可并行占比 $p$
   （提示：两方程消元或直接数值求解），并计算该平台加速比天花板 $1/(1-p)$。
   **交付物**：`journal/amdahl_fit.md`（含公式推导过程）。
3. **batch 心算**（★，约 15 分钟）
   某任务设 `num_envs=2048, num_steps_per_env=32, num_mini_batches=8, num_learning_epochs=5`：
   每次 rollout 多少样本？每个 mini-batch 多少样本？每 iteration 几次梯度更新？
   若显存不足，只改一个参数把显存占用减半，改哪个？为什么？
   **交付物**：写在 `journal/ppo_batch_math.md`。
4. **超参诊断**（★★★，约 2 小时，需 GPU）
   在 Anymal 平地步任务上把 `entropy_coef` 从 0.005 改为 0.05，训练 300 iteration，
   对比动作熵曲线与 mean_reward 曲线的差异，解释高熵系数为何可能「学不动」。
   **交付物**：`journal/entropy_ablation.md` + 两条曲线的截图描述。
5. **导出与独立推理**（★★，约 1 小时，需 GPU）
   用 play.py 导出 ONNX 策略，写一个不依赖 Isaac Lab 的 Python 脚本
   （`onnxruntime` 或 `torch` 加载），随机生成合法维度的观测做推理，验证输出维度与数值范围。
   **交付物**：`journal/onnx_inference.py`。

## 参考答案

<details>
<summary>练习 1：吞吐实测（预期现象与解读模板）</summary>

典型现象：16→256 fps 近似线性（约 16×），256→1024 增速放缓，2048 后接近饱和。
饱和原因候选（用日志分段计时定位）：① 物理 GPU 算力占满（小任务常见）；
② PPO learner 阶段的 kernel launch 串行开销占比上升；③ 显存带宽。
解读模板：「在 <GPU 型号> 上，Ant 任务 fps 从 16 env 的 <a> 增至 4096 env 的 <b>，
加速比 <b/a>，远小于 256× 理想值；按练习 2 拟合 p≈<p̂>，说明串行开销约占 <(1-p̂)>」。
</details>

<details>
<summary>练习 2：Amdahl 反推（推导）</summary>

设 $N_1=16, N_2=4096$，实测加速比 $S_1 = f_1/f_0$，$S_2=f_2/f_0$（$f_0$ 为 $N=1$ 基准，
若没测 $N=1$，可用 $S_2/S_1 = f_2/f_1$ 消去 $f_0$）：

$$\frac{S_2}{S_1} = \frac{(1-p) + p/N_1}{(1-p) + p/N_2}$$

令 $q = 1-p$，解关于 $q$ 的方程：

$$\frac{f_2}{f_1}(q + p/N_2) = q + p/N_1 \;\Rightarrow\; q = \frac{p\,(1/N_1 - r/N_2)}{r - 1},\quad r=\frac{f_2}{f_1}$$

代入 $p = 1-q$ 联立即可（一元一次方程）。天花板 $= 1/q$。
若拟合出 $p \approx 0.995$，天花板约 200×——这就是「GPU 仿真不是无限并行」的量化表达。
</details>

<details>
<summary>练习 3：batch 心算（答案）</summary>

- 每次 rollout 样本数 = $2048 \times 32 = 65{,}536$
- 每 mini-batch = $65{,}536 / 8 = 8{,}192$
- 每 iteration 梯度更新 = $5 \times 8 = 40$ 次
- 显存减半、只改一个参数：`num_mini_batches` 4→8（即题目中的 8 是已减半方案；
  若从 4 起步则改成 8）。因为单次前向/反向的激活显存正比于 mini-batch 大小；
  而 `num_envs`/`num_steps_per_env` 改任何一个都会同时改变 rollout 数据分布
  （不仅是显存问题），`num_learning_epochs` 只改更新次数不改单次显存。
</details>

<details>
<summary>练习 4：entropy_coef 诊断（预期结论）</summary>

`entropy_coef=0.05`（10 倍于默认）时：动作熵曲线**降不下去**（熵正则项在 loss 中权重过大，
梯度被「保持随机」主导），策略长期接近均匀噪声，mean_reward 显著低于基线甚至不增长。
物理解释：PPO 的目标函数 $L = L^{\text{CLIP}} + c_e \cdot H$ 中 $c_e$ 过大等价于
「宁可不拿奖励也要保持探索」，对 locomotion 这类需要精确力控的任务是灾难。
调参原则：熵曲线应在训练中**缓慢下降**；降太快（→0）要调高，不降要调低。
</details>

<details>
<summary>练习 5：ONNX 独立推理（参考脚本）</summary>

```python
import numpy as np
import onnxruntime as ort

sess = ort.InferenceSession("logs/rsl_rl/<run_dir>/exported/policy.onnx")
input_name = sess.get_inputs()[0].name
obs_dim = sess.get_inputs()[0].shape[-1]   # 如 Anymal-C 观测维度

obs = np.random.randn(1, obs_dim).astype(np.float32)
action = sess.run(None, {input_name: obs})[0]

print("输入维度:", obs_dim, "输出维度:", action.shape)
# 验证：输出维度 = 动作维度（关节数）；值域通常在 ±几 内（rsl_rl 输出未过 tanh 的均值）
```

要点：导出的是确定性策略（mean action）；部署时要做观测归一化
（若训练开了 `empirical_normalization`，归一化器会被一并导出或需单独处理——以 play.py
导出内容为准）。
</details>

---

## 延伸阅读

- 论文：[Isaac Gym (Makoviychuk et al., 2021)](https://arxiv.org/abs/2108.10470)、
  [Learning to Walk in Minutes (Rudin et al., 2022)](https://arxiv.org/abs/2201.08117)
- [rsl_rl 源码](https://github.com/leggedrobotics/rsl_rl)（`OnPolicyRunner` 的 batch 组织值得精读）
- [Isaac Lab 文档](https://isaac-sim.github.io/IsaacLab/)（Reinforcement Learning 章节）
- [skrl](https://github.com/Toni-SM/skrl)（想换 SAC/TD3 时的备选框架）